<a href="https://colab.research.google.com/github/jakob-steuer/chemoinformatics_course/blob/main/chemoinformatics_6_pharmacophores.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pharmacophores — the features a protein actually sees

A molecule binds because a handful of its groups sit in the right places: a donor here, an
aromatic ring 5 Å away, a positive charge 7 Å from that. That pattern is a **pharmacophore**, and
it is the oldest idea in rational drug design still in daily use.

| step | what you do |
|---|---|
| 1–2 | find a molecule's features, in 2D and then in 3D |
| 3 | test the classical opioid pharmacophore against four drugs and three controls |
| 4 | **your own compounds** |
| 5 | derive a pharmacophore from two actives by clique detection |
| 6 | screen PubChem with it, and build a docking shortlist |

Steps 4 to 6 are your project work. The companion notebook,
**`chemoinformatics_7_qsar.ipynb`**, asks the same question from the other end: hand a model the
measurements and let *it* find the pattern.

In [ ]:
# Install the packages Colab does not ship with (about a minute).
%pip install -q rdkit py3Dmol networkx

In [ ]:
import os
import itertools
import collections
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
from tqdm.auto import tqdm

from rdkit import Chem, RDConfig, RDLogger, DataStructs
from rdkit.Chem import AllChem, Descriptors, Draw, ChemicalFeatures
from rdkit.Chem.Scaffolds import MurckoScaffold
from rdkit.Chem.rdFingerprintGenerator import GetMorganGenerator
import py3Dmol

warnings.filterwarnings("ignore")
RDLogger.DisableLog("rdApp.*")        # RDKit complains loudly about every unparsable SMILES
plt.rcParams.update({"figure.figsize": (9, 5), "axes.grid": True, "grid.alpha": 0.3,
                     "font.size": 11, "figure.dpi": 110})

print("RDKit ready.")

## What a pharmacophore is

The IUPAC definition:

> A pharmacophore is the **ensemble of steric and electronic features** that is necessary to
> ensure the optimal supramolecular interactions with a specific biological target and to trigger
> (or block) its biological response.

Three things follow, and they are the whole of this section.

**1. It is not a molecule but a pattern** — "a donor here, an aromatic ring 5 Å away, a positive
charge 7 Å from that". Any molecule that can present those features in that arrangement is a
candidate, which is why pharmacophores let you **scaffold-hop**.

**2. It is 3D.** The right features in the wrong geometry do not bind, and geometry needs a
conformer — so a flexible molecule has to be checked in many conformations.

**3. It is a hypothesis**, derived from known actives or from a complex structure, and no better
than that evidence.

### The standard feature types

| feature | meaning | typical chemistry |
|---|---|---|
| 🔵 **Donor** | hydrogen bond donor | –OH, –NH, –NH₂ |
| 🔴 **Acceptor** | hydrogen bond acceptor | C=O, ring N, ether O |
| 🟣 **Aromatic** | aromatic ring (π-stacking) | benzene, pyridine, indole |
| 🟢 **PosIonizable** | positively charged at pH 7 | basic amine, guanidine, amidine |
| 🟠 **NegIonizable** | negatively charged at pH 7 | carboxylate, phosphate, tetrazole |
| 🟡 **Hydrophobe** | greasy contact surface | alkyl chains, halogens, rings |

RDKit assigns all of these automatically. That is where we start.

In [ ]:
# --- feature definitions shipped with RDKit -------------------------------
FEATURE_FACTORY = ChemicalFeatures.BuildFeatureFactory(
    os.path.join(RDConfig.RDDataDir, "BaseFeatures.fdef"))

FAMILIES = ["Donor", "Acceptor", "Aromatic", "PosIonizable", "NegIonizable", "Hydrophobe"]
DEFAULT_SHOWN = ("Donor", "Acceptor", "Aromatic", "PosIonizable", "NegIonizable")
COLORS = {"Aromatic": (0.72, 0.55, 0.90), "Donor": (0.35, 0.65, 0.98),
          "Acceptor": (0.95, 0.40, 0.40), "PosIonizable": (0.40, 0.80, 0.45),
          "NegIonizable": (0.98, 0.60, 0.25), "Hydrophobe": (0.98, 0.85, 0.35)}
COLORS["LumpedHydrophobe"] = COLORS["Hydrophobe"]
HEX = {k: "#%02x%02x%02x" % tuple(int(255 * c) for c in v) for k, v in COLORS.items()}

# An atom can belong to several families at once (a basic NH is both a Donor and a
# PosIonizable). A 2D picture has only one colour per atom, so we need a precedence:
# the family furthest to the right wins.
DRAW_ORDER = ["LumpedHydrophobe", "Hydrophobe", "Aromatic", "Acceptor", "Donor",
              "NegIonizable", "PosIonizable"]


def mol_from(smiles_or_mol):
    """Accept a SMILES string or an RDKit molecule."""
    if isinstance(smiles_or_mol, str):
        return Chem.MolFromSmiles(smiles_or_mol)
    return smiles_or_mol


def smiles_from_pubchem(name):
    """Look up a compound by name in PubChem and return its SMILES (None if not found)."""
    url = ("https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/"
           f"{requests.utils.quote(name)}/property/CanonicalSMILES/TXT")
    try:
        answer = requests.get(url, timeout=20)
        if answer.ok and answer.text.strip():
            return answer.text.strip().split("\n")[0]
    except Exception as error:
        print(f"  PubChem lookup failed for {name!r}: {error}")
    return None


def feature_table(mol):
    """Every pharmacophore feature RDKit finds, as a table."""
    mol = mol_from(mol)
    rows = [{"family": f.GetFamily(), "definition": f.GetType(),
             "atoms": ", ".join(str(a) for a in f.GetAtomIds())}
            for f in FEATURE_FACTORY.GetFeaturesForMol(mol) if f.GetFamily() in FAMILIES]
    return pd.DataFrame(rows)


def feature_counts(named_molecules):
    """Feature profile of several molecules: {name: SMILES or mol} -> table."""
    rows = {}
    for name, mol in named_molecules.items():
        mol = mol_from(mol)
        counts = dict.fromkeys(FAMILIES, 0)
        for f in FEATURE_FACTORY.GetFeaturesForMol(mol):
            if f.GetFamily() in counts:
                counts[f.GetFamily()] += 1
        counts["heavy atoms"] = mol.GetNumHeavyAtoms()
        counts["MW"] = round(Descriptors.MolWt(mol), 1)
        counts["LogP"] = round(Descriptors.MolLogP(mol), 2)
        rows[name] = counts
    return pd.DataFrame(rows).T


def show_features(mol, families=DEFAULT_SHOWN, size=(900, 450), legend="", overlap="precedence"):
    """2D structure with the pharmacophore features highlighted in the colours of the table.

    overlap : how to draw an atom that belongs to several families
              "precedence" (default) — one colour, the strongest family wins (see DRAW_ORDER)
              "rings"                — one coloured ring per family, drawn concentrically
    """
    from IPython.display import SVG
    mol = mol_from(mol)
    feats = [f for f in FEATURE_FACTORY.GetFeaturesForMol(mol) if f.GetFamily() in families]
    drawer = Draw.rdMolDraw2D.MolDraw2DSVG(*size)

    if overlap == "rings":
        colour_map = collections.defaultdict(list)
        for f in feats:
            for atom in f.GetAtomIds():
                colour = COLORS[f.GetFamily()]
                if colour not in colour_map[atom]:
                    colour_map[atom].append(colour)
        drawer.DrawMoleculeWithHighlights(
            Draw.rdMolDraw2D.PrepareMolForDrawing(mol), legend, dict(colour_map), {},
            {a: 0.45 for a in colour_map}, {a: 1 for a in colour_map})
    else:
        feats.sort(key=lambda f: DRAW_ORDER.index(f.GetFamily()))
        atoms, colors, radii = [], {}, {}
        for f in feats:
            for atom in f.GetAtomIds():
                atoms.append(atom)
                colors[atom] = COLORS[f.GetFamily()]      # later families overwrite earlier ones
                radii[atom] = 0.45
        Draw.rdMolDraw2D.PrepareAndDrawMolecule(
            drawer, mol, legend=legend, highlightAtoms=list(dict.fromkeys(atoms)),
            highlightAtomColors=colors, highlightAtomRadii=radii)

    drawer.FinishDrawing()
    return SVG(drawer.GetDrawingText())


def feature_overlaps(mol, families=DEFAULT_SHOWN):
    """Atoms that belong to more than one feature family, and which colour they end up with."""
    mol = mol_from(mol)
    roles = collections.defaultdict(set)
    for f in FEATURE_FACTORY.GetFeaturesForMol(mol):
        if f.GetFamily() in families:
            for atom in f.GetAtomIds():
                roles[atom].add(f.GetFamily())
    rows = [{"atom": atom,
             "element": mol.GetAtomWithIdx(atom).GetSymbol(),
             "H": mol.GetAtomWithIdx(atom).GetTotalNumHs(),
             "aromatic": mol.GetAtomWithIdx(atom).GetIsAromatic(),
             "roles": " + ".join(sorted(families_here)),
             "drawn as": max(families_here, key=DRAW_ORDER.index)}
            for atom, families_here in sorted(roles.items()) if len(families_here) > 1]
    return pd.DataFrame(rows)


def make_conformers(mol, n_conformers=10, seed=0xF00D):
    """Generate and minimise 3D conformers — a pharmacophore only exists in 3D."""
    mol = Chem.AddHs(Chem.Mol(mol_from(mol)))
    if not AllChem.EmbedMultipleConfs(mol, numConfs=n_conformers, randomSeed=seed):
        return None
    AllChem.MMFFOptimizeMoleculeConfs(mol, maxIters=400)
    return mol


def _features_of(mol3d, conf_id, family):
    """(position, atom indices) for every feature of one family in one conformer."""
    found = []
    for f in FEATURE_FACTORY.GetFeaturesForMol(mol3d, confId=conf_id):
        if f.GetFamily() == family:
            position = f.GetPos(conf_id)
            found.append((np.array([position.x, position.y, position.z]),
                          tuple(f.GetAtomIds())))
    return found


def _feature_positions(mol3d, conf_id, family):
    return [position for position, _ in _features_of(mol3d, conf_id, family)]


def feature_distance_range(mol3d, family_a, family_b):
    """Shortest and longest distance between two feature types, over all conformers."""
    found = []
    for conf_id in range(mol3d.GetNumConformers()):
        for pa, pb in itertools.product(_feature_positions(mol3d, conf_id, family_a),
                                        _feature_positions(mol3d, conf_id, family_b)):
            if not np.allclose(pa, pb):
                found.append(float(np.linalg.norm(pa - pb)))
    return (min(found), max(found)) if found else (None, None)


def matches_pharmacophore(mol3d, points, distances, all_matches=False):
    """Does any conformer satisfy all constraints at once?

    points      : feature families, e.g. ["PosIonizable", "Aromatic", "Acceptor"]
    distances   : {(i, j): (min, max)} in Å, indices refer to `points`
    all_matches : False -> (matched, first match found)
                  True  -> a list of every distinct way the query can be satisfied

    A match is a dict with the conformer, the position and the **atoms** of each point,
    and the distances between them.
    """
    collected, seen = [], set()
    if mol3d is None:
        return [] if all_matches else (False, None)

    for conf_id in range(mol3d.GetNumConformers()):
        candidates = [_features_of(mol3d, conf_id, family) for family in points]
        if any(len(c) == 0 for c in candidates):
            continue
        for combination in itertools.product(*candidates):
            found = {}
            for (i, j), (low, high) in distances.items():
                distance = float(np.linalg.norm(combination[i][0] - combination[j][0]))
                if not low <= distance <= high:
                    break
                found[(i, j)] = round(distance, 1)
            else:
                match = {"conformer": conf_id,
                         "positions": [c[0] for c in combination],
                         "atoms": [c[1] for c in combination],
                         "distances": found}
                if not all_matches:
                    return True, match
                key = tuple(match["atoms"])
                if key not in seen:
                    seen.add(key)
                    collected.append(match)
    return collected if all_matches else (False, None)


def describe_group(mol, atom_ids):
    """A short chemical name for the group a feature sits on."""
    atom = mol.GetAtomWithIdx(atom_ids[0])
    if len(atom_ids) > 1:
        return f"ring of {len(atom_ids)} atoms"
    neighbours = list(atom.GetNeighbors())
    has_hydrogen = any(n.GetSymbol() == "H" for n in neighbours)
    next_to_ring = any(n.GetIsAromatic() for n in neighbours)
    if atom.GetSymbol() == "O":
        if any(b.GetBondTypeAsDouble() == 2 for b in atom.GetBonds()):
            carbon = [n for n in neighbours if n.GetSymbol() == "C"][0]
            others = [n.GetSymbol() for n in carbon.GetNeighbors() if n.GetIdx() != atom_ids[0]]
            if "O" in others:
                return "ester C=O"
            return "amide C=O" if "N" in others else "ketone C=O"
        if has_hydrogen:
            return "phenol -OH" if next_to_ring else "alcohol -OH"
        return "ether -O-"
    if atom.GetSymbol() == "N":
        return f"{'aromatic ' if atom.GetIsAromatic() else ''}N with {atom.GetTotalNumHs()} H"
    return atom.GetSymbol()


def match_report(named_molecules, points, distances, n_conformers=10):
    """Which molecules match a query — and which chemical group satisfied each point."""
    rows = []
    for name, molecule in named_molecules.items():
        mol3d = make_conformers(molecule, n_conformers=n_conformers)
        matched, match = matches_pharmacophore(mol3d, points, distances)
        row = {"molecule": name, "matches": matched}
        if matched:
            for family, atom_ids in zip(points, match["atoms"]):
                row[family] = describe_group(mol3d, atom_ids)
            for (i, j), distance in match["distances"].items():
                row[f"{points[i][:4]}-{points[j][:4]} (A)"] = distance
        rows.append(row)
    return pd.DataFrame(rows).fillna("")


# --- 3D viewing ----------------------------------------------------------
def _marker_model(view, positions, colour, radius, opacity=1.0, target=None):
    """Draw spheres at the given positions."""
    if len(positions) == 0:
        return
    where = {} if target is None else {"viewer": target}
    xyz = f"{len(positions)}\n\n" + "\n".join(
        f"He {p[0]:.4f} {p[1]:.4f} {p[2]:.4f}" for p in positions)
    view.addModel(xyz, "xyz", **where)
    view.setStyle({"model": -1},
                  {"sphere": {"radius": radius, "color": colour, "opacity": opacity}}, **where)


def _add_molecule(view, mol3d, conf_id=0, target=None):
    where = {} if target is None else {"viewer": target}
    view.addModel(Chem.MolToMolBlock(mol3d, confId=conf_id), "sdf", **where)
    view.setStyle({"model": -1},
                  {"stick": {"radius": 0.13, "colorscheme": "lightgreyCarbon"}}, **where)


def _dotted_line(view, start, end, target=None, colour="#333333", n_dots=11, radius=0.09):
    """Draw a dotted line between two points."""
    _marker_model(view, [start + (end - start) * t for t in np.linspace(0.1, 0.9, n_dots)],
                  colour, radius, 1.0, target)


def view_features_3d(mol, families=DEFAULT_SHOWN, width=760, height=420, zoom=1.6):
    """One conformer with a coloured sphere on every pharmacophore feature."""
    mol3d = make_conformers(mol, n_conformers=1)
    view = py3Dmol.view(width=width, height=height)
    _add_molecule(view, mol3d)
    for family in families:
        _marker_model(view, _feature_positions(mol3d, 0, family), HEX[family], 0.85, 0.80)
    view.setBackgroundColor("white")
    view.zoomTo()
    view.zoom(zoom)                     # zoomTo fits the bounding sphere, which leaves a long
    return view                         # molecule looking tiny; tighten it up


def view_pharmacophore(mol, points, distances, label="", n_conformers=10,
                       width=760, height=460, view=None, target=None, zoom=1.35):
    """Show the conformer that satisfies a pharmacophore query.

    The matching features are drawn as spheres and the distances between them as dotted
    lines. Pass `view` and `target=(row, column)` to draw into one cell of a viewer grid.
    """
    where = {} if target is None else {"viewer": target}
    mol3d = make_conformers(mol, n_conformers=n_conformers)
    matched, match = matches_pharmacophore(mol3d, points, distances)
    if view is None:
        view = py3Dmol.view(width=width, height=height)

    _add_molecule(view, mol3d, match["conformer"] if matched else 0, target)
    if matched:
        for family, position in zip(points, match["positions"]):
            _marker_model(view, [position], HEX[family], 0.9, 0.80, target)
        for (i, j), distance in match["distances"].items():
            start, end = match["positions"][i], match["positions"][j]
            _dotted_line(view, start, end, target)
            middle = (start + end) / 2
            view.addLabel(f"{distance} A",
                          {"position": {"x": float(middle[0]), "y": float(middle[1]),
                                        "z": float(middle[2])},
                           "fontSize": 11, "fontColor": "black", "backgroundColor": "white",
                           "backgroundOpacity": 0.75, "borderThickness": 0}, **where)
    if label:
        # the leading spaces keep the label clear of the edge of the viewer
        view.addLabel(f"  {label}" if matched else f"  {label}: no match",
                      {"useScreen": True, "x": 10, "y": 10, "fontSize": 14,
                       "fontColor": "black" if matched else "white",
                       "backgroundColor": "white" if matched else "#cc3333",
                       "backgroundOpacity": 0.85}, **where)
    view.setBackgroundColor("white")
    view.zoomTo(**where)
    view.zoom(zoom, **where)
    return view


print("Pharmacophore toolbox ready.")

## Step 1 — Which features does a molecule have?

Let us take **imatinib** (Gleevec), the drug that turned chronic myeloid leukaemia from a fatal
disease into a managed one. The colours are those of the table above.

In [ ]:
IMATINIB = "CC1=C(C=C(C=C1)NC(=O)C2=CC=C(C=C2)CN3CCN(CC3)C)NC4=NC=CC(=N4)C5=CN=CC=C5"

show_features(IMATINIB, legend="imatinib")

In [ ]:
feature_table(IMATINIB).groupby("family").size().to_frame("count")

Read the picture, not the table: the two **blue** NH groups and the **red** ring nitrogens are the
hinge-binding motif that every kinase inhibitor needs; the **purple** rings stack in a mostly
greasy pocket; the **green** piperazine nitrogens are protonated at pH 7 and reach out to solvent,
which is largely why the molecule is soluble enough to be a tablet.

That sentence is a pharmacophore hypothesis. The rest of this notebook makes it testable.

### One atom, several roles

An atom can belong to several families at once, and a 2D picture has only one colour per atom.
Here is where that happens in imatinib:

In [ ]:
feature_overlaps(IMATINIB)

Two cases, and the second is worth knowing:

* **Ring nitrogens are Acceptor + Aromatic** — both true, and independent of each other.
* **The piperazine nitrogens are Donor + PosIonizable with no hydrogen at all** (`H` column:
  zero). RDKit lists trialkylamines as donors deliberately: such a nitrogen is protonated at
  pH 7, and the resulting N–H⁺ *is* a donor. For a basic amine the two labels are not a
  coincidence — they are the same chemistry described twice.

The drawing breaks the tie by precedence, strongest last:

`Hydrophobe → Aromatic → Acceptor → Donor → NegIonizable → PosIonizable`

so a basic NH is drawn green and a pyridine N red, with the rest of its ring still purple.
Use `overlap="rings"` to see every role at once:

In [ ]:
show_features(IMATINIB, overlap="rings", legend="imatinib — every role drawn")

More honest, less legible — the rest of the notebook keeps the single-colour version. The
**numbers** lose nothing: `feature_table` and `feature_counts` count features, not atoms, so a
basic NH is counted under both families. Only the picture has to choose.

### Exercise 1 — Features of a drug you know

Pick any drug and look at its features. `smiles_from_pubchem` fetches the structure by name — the
same PubChem API you will use in the project. Try deliberately different molecules: an antibiotic,
a steroid, a sugar, your project compound.

<details>
<summary>🔍 Something to notice</summary>

```python
for name in ["aspirin", "caffeine", "atorvastatin", "penicillin G", "glucose"]:
    display(show_features(smiles_from_pubchem(name), legend=name))
```

* **Aspirin** has an 🟠 ionizable acid — an anion at pH 7.4, which is why it is absorbed in the
  stomach (where it is neutral) rather than the gut.
* **Glucose** is nothing but 🔵 donors and 🔴 acceptors: no ring, no hydrophobe, no charge. It
  still crosses the blood–brain barrier — through a dedicated transporter, which a features-only
  view cannot know about. Keep that in mind for the QSAR notebook.
* **Atorvastatin** shows how large a real drug's feature set is, and how little a five-descriptor
  model captures.

</details>

## Step 2 — The same features in 3D

Features alone are not a pharmacophore. What matters is **where they are relative to each other**,
and that only exists once the molecule has a conformation.

Drag to rotate: every sphere is one feature, coloured as in the table.

In [ ]:
view_features_3d(IMATINIB)

Now we can measure the thing that makes a pharmacophore a *model*: the distance between features.
Because the molecule is flexible, we generate several conformers and report the range.

In [ ]:
imatinib_3d = make_conformers(IMATINIB, n_conformers=10)

print(f"imatinib, {imatinib_3d.GetNumConformers()} conformers\n")
for family_a, family_b in [("Donor", "Aromatic"), ("Donor", "Acceptor"),
                           ("PosIonizable", "Aromatic"), ("PosIonizable", "Acceptor")]:
    low, high = feature_distance_range(imatinib_3d, family_a, family_b)
    print(f"{family_a:>13s} ... {family_b:<13s} {low:5.1f} – {high:5.1f} Å")

A flexible molecule covers a wide range of distances — imatinib can place its basic nitrogen
anywhere between roughly 4 and 19 Å from an aromatic ring. That flexibility is what lets one molecule
satisfy many different pharmacophores, and it is also why a pharmacophore search must be done over
conformers rather than on a single structure.

## Step 3 — Scaffold hopping: the point of the whole idea

Here is the classic demonstration. These four molecules are all strong **µ-opioid receptor
agonists** — they are painkillers that act on the same receptor:

* **morphine** — the natural product, a rigid five-ring cage
* **pethidine** (meperidine) — a simple piperidine ester
* **methadone** — an open chain, no ring system at all connecting the parts
* **fentanyl** — a different open scaffold again

If you compare them the way a chemist's fingerprint does — by substructure — they have almost
nothing in common. Let us check that first.

In [ ]:
OPIOIDS = {
    "morphine":  "CN1CCC23C4C1CC5=C2C(=C(C=C5)O)OC3C(C=C4)O",
    "pethidine": "CCOC(=O)C1(CCN(CC1)C)C2=CC=CC=C2",
    "methadone": "CCC(=O)C(CC(C)N(C)C)(C1=CC=CC=C1)C2=CC=CC=C2",
    "fentanyl":  "CCC(=O)N(C1CCN(CC1)CCC2=CC=CC=C2)C3=CC=CC=C3",
}
CONTROLS = {
    "aspirin":  "CC(=O)OC1=CC=CC=C1C(=O)O",
    "caffeine": "CN1C=NC2=C1C(=O)N(C(=O)N2C)C",
    "glucose":  "C(C1C(C(C(C(O1)O)O)O)O)O",
}

morgan = GetMorganGenerator(radius=2, fpSize=2048)
reference = morgan.GetFingerprint(Chem.MolFromSmiles(OPIOIDS["morphine"]))

print("Morgan (substructure) similarity to morphine:")
for name, smiles in {**OPIOIDS, **CONTROLS}.items():
    similarity = DataStructs.TanimotoSimilarity(
        reference, morgan.GetFingerprint(Chem.MolFromSmiles(smiles)))
    print(f"   {name:10s} {similarity:.2f}")

Draw.MolsToGridImage([Chem.MolFromSmiles(s) for s in OPIOIDS.values()],
                     legends=list(OPIOIDS), molsPerRow=4, subImgSize=(260, 220))

Now ask the pharmacophore question instead. Since the 1950s the classical model for this family —
the **Beckett–Casy** model — has been two points plus a geometric relation:

* a **protonated nitrogen** (🟢 PosIonizable), which makes a salt bridge with a conserved
  aspartate in the receptor,
* a **flat aromatic ring** (🟣),
* separated by roughly **4–7 Å**.

That is all. Two points and one distance window:

In [ ]:
OPIOID_PHARMACOPHORE = (
    ["PosIonizable", "Aromatic"],        # the two points
    {(0, 1): (4.0, 7.0)},                # N+ ... ring centre, in Å
)

Here it is on morphine. The spheres are the two pharmacophore points and the dotted line is the
distance that satisfies the query — this picture *is* the pharmacophore:

In [ ]:
view_pharmacophore(OPIOIDS["morphine"], *OPIOID_PHARMACOPHORE, label="morphine")

Note what is **not** in that picture: no carbon skeleton, no ring system, no stereochemistry. A
pharmacophore keeps the coloured points and the distances between them and throws the rest of the
molecule away. So any other molecule that can arrange the same points the same way is a candidate —
whatever it is built from.

Let us test that against the four opioids and the three controls. `match_report` also names the
chemical group that satisfied each point, which we will need in a moment.

In [ ]:
match_report({**OPIOIDS, **CONTROLS}, *OPIOID_PHARMACOPHORE)

All four opioids match, at 4.7, 5.7, 6.1 and 5.2 Å. None of the controls does: aspirin and glucose
have no basic nitrogen at all, and caffeine's nitrogens sit *inside* its rings rather than 4–7 Å
away from one.

Here they are side by side — the same two points, four unrelated scaffolds:

In [ ]:
grid = py3Dmol.view(viewergrid=(2, 2), width=860, height=620, linked=False)

for (name, smiles), cell in zip(OPIOIDS.items(), [(0, 0), (0, 1), (1, 0), (1, 1)]):
    view_pharmacophore(smiles, *OPIOID_PHARMACOPHORE, label=name, view=grid, target=cell)

grid.setBackgroundColor("white")
grid

Rotate any of the four and compare. The molecules have almost nothing in common — morphine is a
rigid five-ring cage, methadone an open chain — but each presents a basic nitrogen and an aromatic
ring at the same separation. Morphine does it without effort; the flexible ones (fentanyl,
methadone) must adopt a particular conformation to manage it, which costs energy and is one reason
their potencies differ.

This is the single most important idea here: **a pharmacophore finds molecules that a
substructure search cannot.** Fentanyl is a 0.13-similarity stranger to morphine and yet it
presents the same two features in the same arrangement — which is why it binds the same receptor
(and, being far more potent, why it is so dangerous).

### Exercise 2 — Would a third point be better?

Two points is a thin model. All four molecules also contain a **hydrogen bond acceptor**, so the
obvious improvement is to add it as a third point:

```python
THREE_POINT = (
    ["PosIonizable", "Aromatic", "Acceptor"],
    {(0, 1): (4.0, 7.0),      # N+ ... ring centre
     (0, 2): (4.0, 9.0),      # N+ ... acceptor
     (1, 2): (2.5, 7.0)},     # ring centre ... acceptor
)
```

Run `match_report(OPIOIDS, *THREE_POINT)` and read it **column by column** before going on. Do all
four still match — and did the third point land on the same kind of group each time?

<details>
<summary>🔍 Stage 1 — all four match, and that is the problem</summary>

```python
THREE_POINT = (
    ["PosIonizable", "Aromatic", "Acceptor"],
    {(0, 1): (4.0, 7.0), (0, 2): (4.0, 9.0), (1, 2): (2.5, 7.0)},
)

match_report(OPIOIDS, *THREE_POINT)
```

| molecule | Acceptor | N⁺–acc (Å) | ring–acc (Å) |
|---|---|---|---|
| morphine | **phenol –OH** | 7.2 | **2.8** |
| pethidine | **ether –O–** | 5.0 | 4.0 |
| methadone | **ketone C=O** | 4.6 | 4.6 |
| fentanyl | **amide C=O** | 4.7 | 5.0 |

All four match, and the model is nonsense: the third point is satisfied by a phenol, an ether, a
ketone and an amide. Morphine gives it away numerically — its acceptor sits 2.8 Å from the ring
centre (a substituent *on* the ring) and 7.2 Å from the nitrogen, where the others sit 4–5 Å from
both.

A query that "works" because each molecule found its own unrelated way to satisfy it has told you
nothing. **Always look at which atoms matched, not just whether something matched.**

</details>

<details>
<summary>🔍 Stage 2 — tighten the windows and it becomes consistent</summary>

The numbers suggest the fix: exclude acceptors sitting on the ring, and require a similar distance
to both other points.

```python
THREE_POINT_TIGHT = (
    ["PosIonizable", "Aromatic", "Acceptor"],
    {(0, 1): (4.0, 7.0), (0, 2): (4.5, 6.5), (1, 2): (3.5, 6.0)},
)

match_report({**OPIOIDS, **CONTROLS}, *THREE_POINT_TIGHT)
```

Now the assignments line up — morphine through its **C6 alcohol**, methadone and fentanyl through
their **carbonyls**, pethidine through its ester — and the controls still fail.

Except for one detail, already visible in the table: pethidine is reported as matching through the
ester's **ether –O–**, not its C=O. Ask for *every* way the query can be satisfied:

```python
for name, smiles in OPIOIDS.items():
    mol3d = make_conformers(smiles, n_conformers=10)
    groups = {describe_group(mol3d, m["atoms"][2])
              for m in matches_pharmacophore(mol3d, *THREE_POINT_TIGHT, all_matches=True)}
    print(f"{name:10s} {sorted(groups)}")
```

Morphine, methadone and fentanyl have one option each. **Pethidine has two**: its ester C=O and
the ester's own –O–, about 2.2 Å apart. No distance-based query can separate two atoms that close;
only chemistry can say a carbonyl is a good acceptor and an ester's ether oxygen a poor one. That
is a hard limit of the method, not a parameter to tune.

</details>

<details>
<summary>🔍 Stage 3 — now check it against what is known</summary>

We have a model that fits all four actives, rejects the controls, and assigns a comparable group
in each molecule. It looks like a result.

Now look up what medicinal chemistry says about morphine. The hydroxyl that matters for µ-opioid
binding is the **phenol** — the one our tightened query specifically excluded. The C6 position is
the one you can acetylate (giving heroin) without losing activity.

So the model is self-consistent and still encodes the wrong interaction for one of its four
molecules. Every step was reasonable; only knowing the chemistry, or docking the ligands, would
have caught it.

Three things for your project:

1. **Matching is not evidence.** Report which atoms matched, every time.
2. **Tightening a query until it behaves means imposing what you already believed** — legitimate
   as a hypothesis, not as a finding.
3. **A pharmacophore cannot be validated against itself.** It needs a structure, a mutation or an
   assay.

Good material for your "three reasons our conclusion could be wrong" slide — and the reason the
main text sticks to the two-point model, which claims less and is true.

</details>

### Exercise 3 — How picky is our query, really?

Finding your known actives is the easy half; **not** finding everything else is the hard half. Run
the query against a few hundred ordinary drug-like molecules.

```python
library = pd.read_csv("https://deepchemdata.s3-us-west-1.amazonaws.com/datasets/BBBP.csv")
library = library.sample(150, random_state=0)
library["mol"] = library["smiles"].apply(Chem.MolFromSmiles)
library = library.dropna(subset=["mol"])

hits = 0
for mol in library["mol"]:
    # ... build conformers (3 is enough) and test the pharmacophore
    pass

print(f"{hits} / {len(library)} molecules match")
```

<details>
<summary>🔍 Solution and what it means</summary>

```python
hits = 0
for mol in library["mol"]:
    mol3d = make_conformers(mol, n_conformers=3)
    if matches_pharmacophore(mol3d, *OPIOID_PHARMACOPHORE)[0]:
        hits += 1

print(f"{hits} / {len(library)} molecules match ({hits/len(library)*100:.0f} %)")
```

About half a minute, and **56 of 148 — 38 %**. More than a third of an arbitrary drug library
satisfies our "opioid" pharmacophore. That is the central limitation of the method: most drugs
contain a ring and a basic nitrogen somewhere, flexible molecules can reach almost any distance
given enough conformers, and nothing in the query says the rest of the molecule has to **fit**.

Adding points helps, at a price:

| query | hits in 148 drugs |
|---|---|
| two-point | 38 % |
| three-point, loose windows | 22 % |
| three-point, tightened windows | 11 % |

— but Exercise 2 showed what those extra points can cost in meaning. Even at 11 % a pharmacophore
is a *filter*, not an answer. Real screening adds **excluded volumes** (regions where protein atoms
sit, so the ligand must not) and a shape constraint, then hands the survivors to docking.

</details>

## Step 4 — Your project compounds

From here on, this notebook is your own work. Both tracks can be opened up with the tools above, and in
both cases the interesting answer is usually **what the pharmacophore does *not* explain**.

**Selectivity track** — *one drug, two related proteins; why does it bind one and not the other?*
Take a known binder of each protein and compare their feature patterns. If the two sets of actives
need different features, you have a hypothesis to check against the structures.

**Resistance track** — *wild-type versus clinical mutant; why does the drug stop working?* The drug
does not change, so start from one that still works: most clinically important resistance mutations
have a next-generation inhibitor designed to beat them. Compare the generations — what did the
chemists add, remove or rigidify?

The cell below does this for two textbook pairs.

In [ ]:
PAIRS = {
    "EGFR, T790M resistance": ["gefitinib", "osimertinib"],   # 1st vs 3rd generation
    "ABL vs SRC selectivity": ["imatinib", "dasatinib"],      # selective vs promiscuous
}

for title, drugs in PAIRS.items():
    molecules = {}
    for drug in drugs:
        smiles = smiles_from_pubchem(drug)
        if smiles:
            molecules[drug] = smiles
    if len(molecules) < 2:
        print(f"{title}: could not fetch both compounds, skipping")
        continue

    fingerprints = [morgan.GetFingerprint(Chem.MolFromSmiles(s)) for s in molecules.values()]
    similarity = DataStructs.TanimotoSimilarity(*fingerprints)
    print(f"\n### {title}  —  Morgan similarity {similarity:.2f}")
    display(feature_counts(molecules))
    display(Draw.MolsToGridImage([Chem.MolFromSmiles(s) for s in molecules.values()],
                                 legends=list(molecules), molsPerRow=2, subImgSize=(380, 300)))

### Exercise 4 — Read the comparisons

For each pair: did the number of features change much? If not, what *did* change — and what would
you need in order to explain it properly? Then run the same comparison on your own compounds and
write down a one-sentence hypothesis.

<details>
<summary>🔍 Discussion</summary>

**The features barely move.** Gefitinib and osimertinib differ by one donor and two acceptors;
imatinib and dasatinib have near-identical profiles. On feature counts alone you would call each
pair interchangeable. They are not — one member of each pair does something the other cannot.

**What changed is chemistry and geometry the feature list does not encode.**

* **Osimertinib vs gefitinib (EGFR T790M).** Osimertinib carries an **acrylamide** that forms a
  *covalent* bond to a cysteine in the pocket. No standard feature family describes "Michael
  acceptor that will react with a nearby cysteine" — the irreversible chemistry, the entire point
  of the molecule, is invisible here.
* **Dasatinib vs imatinib (selectivity).** Imatinib binds a particular *inactive* kinase
  conformation that few kinases adopt; dasatinib binds the active one, which many share. The
  selectivity is a property of the **protein's** conformational preference, not of the ligand's
  features at all.

A third, classic case you could add yourself: **ponatinib vs imatinib** for the ABL T315I
gatekeeper mutation, where the answer is a rigid carbon–carbon triple bond — a pure **shape**
change, which is exactly what a feature-count table throws away.

All three need the protein: a structure, the pocket shape, docking. That is the honest conclusion
of this notebook, and a good candidate for your "three reasons our conclusion could be wrong" slide:

> A pharmacophore is a *necessary* condition for binding, not a sufficient one. It can tell you
> why a molecule **cannot** bind. It cannot, on its own, tell you why one of two molecules with
> the same features binds better.

</details>

### Your own compounds

Edit the names (or paste SMILES) and run.

In [ ]:
MY_COMPOUNDS = ["imatinib", "ponatinib"]        # <-- your drug / your two drugs

my_molecules = {}
for name in MY_COMPOUNDS:
    smiles = name if Chem.MolFromSmiles(name) else smiles_from_pubchem(name)
    if smiles is None:
        print(f"could not find {name!r} in PubChem — paste the SMILES instead")
    else:
        my_molecules[name] = smiles

display(feature_counts(my_molecules))

for name, smiles in my_molecules.items():
    display(show_features(smiles, legend=name))
    mol3d = make_conformers(smiles, n_conformers=10)
    low, high = feature_distance_range(mol3d, "Donor", "Aromatic")
    if low is not None:
        print(f"{name}: donor–aromatic distance spans {low:.1f}–{high:.1f} Å "
              f"over {mol3d.GetNumConformers()} conformers")

### Exercise 5 — Your compound in 3D

The 2D picture tells you *which* features exist; only the 3D one tells you whether a protein could
use them together. Rotate your compound and ask: which features point the same way, and which sit
on opposite faces? A pocket can only reach one side at a time.

In [ ]:
first = list(my_molecules)[0]
view_features_3d(my_molecules[first])

Then measure the distances that define your own pharmacophore hypothesis — the pairs you believe
the protein actually uses:

```python
mol3d = make_conformers(my_molecules[first], n_conformers=10)

for a, b in [("Donor", "Aromatic"), ("PosIonizable", "Acceptor")]:
    low, high = feature_distance_range(mol3d, a, b)
    if low is not None:
        print(f"{a:>13s} ... {b:<13s} {low:5.1f} – {high:5.1f} Å")
```

## Step 5 — Deriving a pharmacophore from known actives

So far we *guessed* a pharmacophore and then tested it. The real workflow runs the other way: you
have several molecules known to hit the same target and you want the model they all satisfy. That
is **pharmacophore mapping**.

The classical algorithm is **clique detection** (the method behind DISCO):

1. Take one conformer of molecule A and one of molecule B.
2. Build a **correspondence graph**. Each node is a *pair* of same-type features — "ring 2 of A
   could play the role of ring 1 of B". Two nodes are joined when they are mutually consistent:
   the distance between the features inside A matches the distance between their partners in B,
   within a tolerance.
3. A **clique** — a set of nodes all joined to each other — is a set of assignments consistent
   with one another. The **largest** clique is the biggest common pharmacophore.
4. Repeat over conformer pairs, keep the best.

Step 4 is where the cost lives: *n* molecules × *k* conformers means kⁿ combinations, and clique
detection is NP-hard. That is why **GASP** exists — a genetic algorithm that searches molecule and
conformer space together instead of enumerating it, trading the guarantee of the best answer for
the ability to handle flexible molecules and larger sets. We stay with clique detection on two
molecules, which is exact and fast enough to watch.

### The same construction, on two real molecules

The lecture showed this on an abstract pair of graphs. Here it is on two drugs.

**Sulfanilamide** (the first sulfa antibacterial) and **dapsone** (used against leprosy) both
inhibit the same bacterial enzyme, dihydropteroate synthase, and both do it the same way: by
impersonating its natural substrate **PABA** (4-aminobenzoic acid). So they *should* have a
pharmacophore in common, and we can watch the algorithm find it.

To keep the figure identical in shape to the lecture slide we use only **Donor** and **Acceptor**
features, and **one conformer** each. Conveniently, that gives both molecules exactly two donors
and two acceptors — the slide's D1, D2, A1, A2 against d1, d2, a1, a2.

In [ ]:
SULFANILAMIDE = "Nc1ccc(cc1)S(N)(=O)=O"
DAPSONE = "Nc1ccc(cc1)S(=O)(=O)c1ccc(N)cc1"


def labelled_features(smiles, uppercase=True, families=("Donor", "Acceptor")):
    """One conformer, features named D1, D2, A1, A2 (or d1, d2, a1, a2)."""
    mol = Chem.AddHs(Chem.MolFromSmiles(smiles))
    AllChem.EmbedMolecule(mol, randomSeed=0xF00D)
    AllChem.MMFFOptimizeMolecule(mol)

    found = {family: [] for family in families}
    for feature in FEATURE_FACTORY.GetFeaturesForMol(mol):
        if feature.GetFamily() in found:
            position = feature.GetPos()
            found[feature.GetFamily()].append(
                (np.array([position.x, position.y, position.z]), feature.GetAtomIds()[0]))

    labelled = []
    for family, letter in zip(families, "DA"):
        for number, (position, atom) in enumerate(found[family], start=1):
            name = f"{letter}{number}" if uppercase else f"{letter.lower()}{number}"
            labelled.append((name, family, position, atom))
    return labelled


def draw_labelled(smiles, features, size=(430, 300)):
    """The structure with every feature named on the atom it sits on."""
    from IPython.display import SVG
    mol = Chem.MolFromSmiles(smiles)
    colours = {}
    for name, family, _, atom in features:
        mol.GetAtomWithIdx(atom).SetProp("atomNote", name)
        colours[atom] = COLORS[family]
    drawer = Draw.rdMolDraw2D.MolDraw2DSVG(*size)
    drawer.drawOptions().annotationFontScale = 0.9
    Draw.rdMolDraw2D.PrepareAndDrawMolecule(
        drawer, mol, highlightAtoms=list(colours), highlightAtomColors=colours,
        highlightAtomRadii={atom: 0.42 for atom in colours})
    drawer.FinishDrawing()
    return SVG(drawer.GetDrawingText())


features_1 = labelled_features(SULFANILAMIDE, uppercase=True)
features_2 = labelled_features(DAPSONE, uppercase=False)

display(draw_labelled(SULFANILAMIDE, features_1))
display(draw_labelled(DAPSONE, features_2))

🔵 donors, 🔴 acceptors. Sulfanilamide carries an aniline NH₂ (D1), a sulfonamide NH₂ (D2) and two
sulfonyl oxygens (A1, A2); dapsone is symmetric, so both its donors are aniline NH₂ groups.

**Step 1 — each molecule becomes a graph.** The features are the nodes and the distances between
them are the edge labels. Nothing else about the molecule survives this step.

In [ ]:
import networkx as nx


def distance_graph(features):
    """Features as nodes, the distance between them on every edge."""
    graph = nx.Graph()
    for name, family, _, _ in features:
        graph.add_node(name, family=family)
    for (name_a, _, position_a, _), (name_b, _, position_b, _) in itertools.combinations(features, 2):
        graph.add_edge(name_a, name_b, distance=float(np.linalg.norm(position_a - position_b)))
    return graph


def draw_graph(ax, graph, title, highlight=()):
    """Lay the nodes out so that distances on the page reflect the real ones."""
    spacing = {a: {b: graph[a][b]["distance"] for b in graph if b != a} for a in graph}
    positions = nx.kamada_kawai_layout(graph, dist=spacing)
    bold = [e for e in graph.edges if e[0] in highlight and e[1] in highlight]
    plain = [e for e in graph.edges if e not in bold]

    nx.draw_networkx_edges(ax=ax, G=graph, pos=positions, edgelist=plain, width=1.2,
                           edge_color="0.75")
    nx.draw_networkx_edges(ax=ax, G=graph, pos=positions, edgelist=bold, width=3.5,
                           edge_color="#222222")
    nx.draw_networkx_nodes(ax=ax, G=graph, pos=positions, node_size=900,
                           node_color=[HEX[graph.nodes[n]["family"]] for n in graph],
                           edgecolors=["black" if n in highlight else "0.6" for n in graph],
                           linewidths=[2.5 if n in highlight else 1.0 for n in graph])
    nx.draw_networkx_labels(ax=ax, G=graph, pos=positions, font_size=11, font_weight="bold")
    labels = nx.get_edge_attributes(graph, "distance")
    if labels:
        nx.draw_networkx_edge_labels(
            ax=ax, G=graph, pos=positions, font_size=9, rotate=False,
            bbox={"boxstyle": "round,pad=0.15", "fc": "white", "ec": "none"},
            edge_labels={e: f"{d:.1f}" for e, d in labels.items()})
    ax.set_title(title, fontsize=12)
    ax.margins(0.18)
    ax.set_axis_off()


graph_1, graph_2 = distance_graph(features_1), distance_graph(features_2)

figure, axes = plt.subplots(1, 2, figsize=(13, 5))
draw_graph(axes[0], graph_1, "sulfanilamide — distances in Å")
draw_graph(axes[1], graph_2, "dapsone — distances in Å")
plt.tight_layout()
plt.show()

**Step 2 — every pair of features of the same type becomes a node.** Two donors against two donors
and two acceptors against two acceptors gives 2×2 + 2×2 = **8 nodes**: D1d1, D1d2, D2d1, D2d2,
A1a1, A1a2, A2a1, A2a2. Each one is a *proposal* — "D1 of sulfanilamide could play the role of d1
of dapsone".

**Step 3 — join two proposals when they agree.** Node D1d2 and node A1a2 are compatible if the
D1–A1 distance in sulfanilamide matches the d2–a2 distance in dapsone:

* D1–A1 = 6.6 Å, d2–a2 = 6.5 Å → differ by 0.1 Å → **edge**
* D2–A1 = 2.5 Å, d1–a1 = 6.5 Å → differ by 4.0 Å → **no edge**

A proposal cannot be joined to one that reuses the same feature, so D1d1 and D1d2 are never
connected.

In [ ]:
def correspondence_graph(features_a, features_b, tolerance=1.0):
    """Nodes = same-type feature pairs; edges = pairs whose internal distances agree."""
    graph = nx.Graph()
    for (name_a, family_a, _, _), (name_b, family_b, _, _) in itertools.product(features_a, features_b):
        if family_a == family_b:
            graph.add_node(f"{name_a}{name_b}", family=family_a, a=name_a, b=name_b)

    positions_a = {name: position for name, _, position, _ in features_a}
    positions_b = {name: position for name, _, position, _ in features_b}

    for node_1, node_2 in itertools.combinations(list(graph), 2):
        a1, b1 = graph.nodes[node_1]["a"], graph.nodes[node_1]["b"]
        a2, b2 = graph.nodes[node_2]["a"], graph.nodes[node_2]["b"]
        if a1 == a2 or b1 == b2:                       # a feature cannot play two roles
            continue
        distance_a = float(np.linalg.norm(positions_a[a1] - positions_a[a2]))
        distance_b = float(np.linalg.norm(positions_b[b1] - positions_b[b2]))
        if abs(distance_a - distance_b) <= tolerance:
            # how well the two distances agree — used to rank equally large cliques
            graph.add_edge(node_1, node_2, mismatch=abs(distance_a - distance_b))
    return graph


def maximum_cliques(graph):
    """Every largest clique, the best-fitting one first.

    Cliques of equal size are ranked by how well their distances agree; the node names
    settle anything still tied, so the answer does not depend on enumeration order.
    """
    cliques = list(nx.find_cliques(graph))
    biggest = max(len(clique) for clique in cliques)
    largest = [sorted(clique) for clique in cliques if len(clique) == biggest]

    def total_mismatch(clique):
        return sum(graph[x][y]["mismatch"] for x, y in itertools.combinations(clique, 2))

    return sorted(largest, key=lambda clique: (round(total_mismatch(clique), 3), clique))


correspondence = correspondence_graph(features_1, features_2, tolerance=1.0)
largest_cliques = maximum_cliques(correspondence)
largest_clique = largest_cliques[0]

print(f"{correspondence.number_of_nodes()} nodes, {correspondence.number_of_edges()} edges")
print(f"{len(largest_cliques)} maximum cliques of {len(largest_clique)} features:")
for clique in largest_cliques:
    print("   ", clique)
print(f"\nkeeping {largest_clique}")

figure, ax = plt.subplots(figsize=(7, 6))
positions = nx.circular_layout(correspondence)
bold = [e for e in correspondence.edges if e[0] in largest_clique and e[1] in largest_clique]
nx.draw_networkx_edges(ax=ax, G=correspondence, pos=positions, width=1.0, edge_color="0.8",
                       edgelist=[e for e in correspondence.edges if e not in bold])
nx.draw_networkx_edges(ax=ax, G=correspondence, pos=positions, edgelist=bold, width=3.5,
                       edge_color="#222222")
nx.draw_networkx_nodes(ax=ax, G=correspondence, pos=positions, node_size=1300,
                       node_color=[HEX[correspondence.nodes[n]["family"]] for n in correspondence],
                       edgecolors=["black" if n in largest_clique else "0.7"
                                   for n in correspondence],
                       linewidths=[2.5 if n in largest_clique else 1.0 for n in correspondence])
nx.draw_networkx_labels(ax=ax, G=correspondence, pos=positions, font_size=10, font_weight="bold")
ax.set_title(f"correspondence graph — maximum clique = {len(largest_clique)}", fontsize=12)
ax.margins(0.22)
ax.set_axis_off()
plt.show()

**Step 4 — the largest clique is the pharmacophore.** Four cliques of three nodes are tied for
largest, and the code keeps the best-fitting one: **A1a1, A2a2 and D1d1**. Read back to the
molecules, that says:

| sulfanilamide | dapsone | distance |
|---|---|---|
| A1 (sulfonyl O) | a1 (sulfonyl O) | — |
| A2 (sulfonyl O) | a2 (sulfonyl O) | A1–A2 2.5 Å ≈ a1–a2 2.5 Å |
| D1 (**aniline** NH₂) | d1 (**aniline** NH₂) | D1–A1 6.6 Å ≈ d1–a1 6.5 Å |

An aromatic amine with two hydrogen bond acceptors about 6.5 Å away — exactly the arrangement
PABA presents to the enzyme, and exactly why both drugs work.

Look also at what was **left out**. Sulfanilamide's D2, its sulfonamide NH₂, is in no clique:
dapsone is a sulfone and simply has no NH there. The algorithm discarded the one feature the two
molecules do not share, without being told which that was.

> **Why are there four tied cliques?** Look at the two molecules again and count the ways the same
> three points can be assigned. (Hint: which atoms in each molecule are interchangeable?)

<details>
<summary>🔍 Answer</summary>

Two independent symmetries, 2 × 2 = 4:

* the **two sulfonyl oxygens** are equivalent in both molecules — A1→a1/A2→a2 works just as well
  as A1→a2/A2→a1;
* **dapsone is symmetric** about its sulfone, so its two aniline NH₂ groups are interchangeable —
  D1 can map onto d1 or d2.

All four give identical distances, so this is one pharmacophore counted four times, not four
different answers. Ties are not always this harmless: when the tied cliques are *not* related by
symmetry they are genuinely different alignments, which is why the code ranks them by how well
their distances agree instead of taking whichever it meets first.

</details>

That is the whole method. Everything below is the same four steps, written to loop over conformers
and all six feature families instead of one conformer and two.

In [ ]:
import networkx as nx


def _correspondence_graph(features_a, features_b, tolerance, min_separation):
    """Nodes = pairs of same-type features; edges = pairs whose internal distances agree."""
    nodes = [(i, j) for i, (fa, _) in enumerate(features_a)
             for j, (fb, _) in enumerate(features_b) if fa == fb]
    graph = nx.Graph()
    graph.add_nodes_from(nodes)
    for (i, j), (k, l) in itertools.combinations(nodes, 2):
        if i == k or j == l:                       # a feature cannot play two roles
            continue
        distance_a = float(np.linalg.norm(features_a[i][1] - features_a[k][1]))
        distance_b = float(np.linalg.norm(features_b[j][1] - features_b[l][1]))
        if distance_a < min_separation or distance_b < min_separation:
            continue                               # the two points must be distinct in space
        if abs(distance_a - distance_b) <= tolerance:
            # how well the two distances agree — used to rank equally large cliques
            graph.add_edge((i, j), (k, l), mismatch=abs(distance_a - distance_b))
    return graph


def _features_with_family(mol3d, conf_id, families):
    return [(family, position) for family in families
            for position in _feature_positions(mol3d, conf_id, family)]


def common_pharmacophore(molecule_a, molecule_b, tolerance=1.5, min_separation=2.0,
                         n_conformers=5, families=DEFAULT_SHOWN, report=True):
    """Largest set of features two molecules can present in the same geometry.

    Several cliques are usually tied for largest. We keep the one whose distances agree
    best, and let the clique itself settle any remaining tie, so the result is the same
    on every machine rather than whichever one the search happened to reach first.
    """
    mol_a = make_conformers(molecule_a, n_conformers=n_conformers)
    mol_b = make_conformers(molecule_b, n_conformers=n_conformers)

    biggest, candidates = 0, []
    for conf_a in range(mol_a.GetNumConformers()):
        features_a = _features_with_family(mol_a, conf_a, families)
        for conf_b in range(mol_b.GetNumConformers()):
            features_b = _features_with_family(mol_b, conf_b, families)
            graph = _correspondence_graph(features_a, features_b, tolerance, min_separation)
            for clique in nx.find_cliques(graph):
                if len(clique) < biggest:
                    continue
                if len(clique) > biggest:
                    biggest, candidates = len(clique), []
                mismatch = sum(graph[x][y]["mismatch"]
                               for x, y in itertools.combinations(clique, 2))
                candidates.append({"clique": sorted(clique), "features_a": features_a,
                                   "features_b": features_b, "conformers": (conf_a, conf_b),
                                   "mismatch": round(mismatch, 3)})
    if not candidates:
        return None

    candidates.sort(key=lambda c: (c["mismatch"], c["clique"]))
    if report:
        print(f"{len(candidates)} maximum cliques of {biggest} features "
              f"(distance mismatch {candidates[0]['mismatch']:.2f}–"
              f"{candidates[-1]['mismatch']:.2f} Å) — keeping the best-fitting one")
    return candidates[0]


def as_query(mapping, padding=0.75):
    """Turn a common pharmacophore into a (points, distances) query."""
    clique = mapping["clique"]
    points = [mapping["features_a"][i][0] for i, _ in clique]
    distances = {}
    for (p, (i, j)), (q, (k, l)) in itertools.combinations(list(enumerate(clique)), 2):
        distance_a = float(np.linalg.norm(mapping["features_a"][i][1] - mapping["features_a"][k][1]))
        distance_b = float(np.linalg.norm(mapping["features_b"][j][1] - mapping["features_b"][l][1]))
        distances[(p, q)] = (round(min(distance_a, distance_b) - padding, 1),
                             round(max(distance_a, distance_b) + padding, 1))
    return points, distances


print("Pharmacophore mapping ready.")

Try it on two opioids that share nothing structurally — morphine and fentanyl, Tanimoto 0.13:

In [ ]:
mapping = common_pharmacophore(OPIOIDS["morphine"], OPIOIDS["fentanyl"])
derived_points, derived_distances = as_query(mapping)

print(f"{len(derived_points)} common features: {derived_points}\n")
for (p, q), (low, high) in derived_distances.items():
    print(f"  {derived_points[p]:>13s} ... {derived_points[q]:<13s} {low:4.1f} – {high:4.1f} Å")

Three points — a donor, an aromatic ring and an acceptor — at roughly 4–6 Å from each other, out
of 38 equally large candidates. The donor here is morphine's **basic nitrogen**, the same atom
that Step 3 called a PosIonizable: as in Step 1, one atom carries both labels, so this is the
model we assumed in Step 3, rediscovered from the molecules alone and wearing a different name.

That is the result you want from mapping: not a prettier picture, but a model you did not put in
by hand.

Note what it does *not* give you: any guarantee that the shared arrangement is the one the receptor
reads. Two molecules always share *something*, and here 38 arrangements were tied — we kept the
one whose distances agree best, which is a geometric argument, not a biological one. Exercise 2
applies with full force.

### Exercise 6 — Map your own actives

Run `common_pharmacophore` on two compounds known to hit your target — your drug and a second
known binder, or the two generations from the resistance track.

```python
first, second = list(my_molecules)[:2]

mapping = common_pharmacophore(my_molecules[first], my_molecules[second])
my_points, my_distances = as_query(mapping)
```

Then ask:

1. How many points come out? Fewer than three and the model is too weak to screen with.
2. Are they chemically sensible for your target — does a kinase hinge binder really produce a
   donor and an acceptor a few Å apart?
3. Rerun with `tolerance=1.0` and `tolerance=2.5`. How stable is the answer? A model that changes
   completely with the tolerance is telling you the two molecules do not have much in common.

## Step 6 — Searching PubChem with your pharmacophore

A pharmacophore is worth having because you can **screen** with it. The full version of this is
virtual screening over millions of compounds; the version that fits in a notebook is a few hundred,
which is enough to hand you a short list to dock next to your own ligand.

We take the candidates from PubChem's 2D similarity search and keep the ones that satisfy the
query. Two requests and about a minute of conformer generation.

In [ ]:
def pubchem_similar(smiles, threshold=80, max_records=100, timeout=60, attempts=3):
    """2D-similarity search in PubChem. Returns a table of CID, name and SMILES."""
    url = ("https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/fastsimilarity_2d/smiles/"
           "property/SMILES,Title/JSON")
    for attempt in range(1, attempts + 1):
        try:
            answer = requests.post(url, data={"smiles": smiles, "Threshold": threshold,
                                              "MaxRecords": max_records}, timeout=timeout)
            answer.raise_for_status()
            return pd.DataFrame(answer.json()["PropertyTable"]["Properties"])
        except Exception as error:          # PubChem is busy often enough to be worth retrying
            print(f"  PubChem attempt {attempt}/{attempts} failed: {error}")
            if attempt == attempts:
                raise
            time.sleep(3 * attempt)


def screen(candidates, points, distances, n_conformers=3, max_heavy_atoms=60):
    """Keep the candidates whose conformers satisfy the query."""
    kept = []
    for _, row in tqdm(list(candidates.iterrows()), desc="screening", leave=False):
        mol = Chem.MolFromSmiles(row["SMILES"])
        if mol is None or mol.GetNumHeavyAtoms() > max_heavy_atoms:
            continue
        if matches_pharmacophore(make_conformers(mol, n_conformers), points, distances)[0]:
            kept.append({"CID": row["CID"], "name": row["Title"], "SMILES": row["SMILES"]})
    return pd.DataFrame(kept)


print("PubChem screening ready.")

In [ ]:
candidates = pubchem_similar(OPIOIDS["morphine"], threshold=80, max_records=60)
print(f"PubChem returned {len(candidates)} compounds")

hits = screen(candidates, derived_points, derived_distances)
print(f"{len(hits)} of them satisfy the derived pharmacophore\n")
display(hits.head(12))

Draw.MolsToGridImage([Chem.MolFromSmiles(s) for s in hits["SMILES"].head(8)],
                     legends=list(hits["name"].head(8)), molsPerRow=4, subImgSize=(250, 200))

The hit list is full of real opioids — naloxone, hydromorphone, oxycodone, buprenorphine,
etorphine — alongside a few plant alkaloids that happen to carry the same arrangement.

Be clear about what this does and does not show. We searched PubChem by **2D similarity to
morphine**, so the candidates were morphine-like before the pharmacophore saw them; the filter
confirms rather than discovers. To find genuinely new scaffolds you need a large, *diverse* library
— ChEMBL, ZINC, Enamine — and that is real virtual screening, outside this notebook. What you have
here is the practical version for the project: **a reproducible way to collect compounds related to
your ligand and narrow them to the ones that could present the same features.**

### Exercise 7 — Build your docking shortlist

Run the search for your own compound and your own derived pharmacophore:

```python
candidates = pubchem_similar(my_molecules[first], threshold=80, max_records=60)
hits = screen(candidates, my_points, my_distances)
hits.to_csv("my_pharmacophore_hits.csv", index=False)
```

Then curate it, because an automated list is not a result:

1. **Look at the structures.** Salts, isotopically labelled versions and your own compound will be
   in there.
2. **Check a few in the literature.** Are any actually known binders of your target? Known
   *inactives* are even more valuable — they are your negative controls for docking.
3. **Pick 3–5 to carry forward.** Aim for a spread: one close analogue, one that is structurally
   different, one you expect to fail.

That shortlist, with your reasons for each choice, is exactly the kind of slide that makes a
10-minute presentation defensible. **Write the three to five names down** — the QSAR notebook
asks for them in its configuration cell, and predicts their affinity against both of your proteins.

# Where this goes next

| from this notebook | use it for |
|---|---|
| `show_features`, `feature_counts` | describe your ligand and its analogues in feature terms |
| `common_pharmacophore`, `as_query` | derive a model from your actives instead of guessing one |
| `matches_pharmacophore`, `match_report` | test it — and check *which atoms* matched |
| `pubchem_similar`, `screen` | build a reproducible shortlist of compounds to dock |

**The honest limit of all of it.** A pharmacophore is a *necessary* condition for binding, not a
sufficient one. It can tell you why a molecule **cannot** bind. It cannot tell you why one of two
molecules with the same features binds better — that needs the protein: its pocket shape, the
conformation it prefers, the chemistry it can be made to react with.

Two ways on, and you will use both:

* **`chemoinformatics_7_qsar.ipynb`** — let measured data find the pattern instead, on your own
  target, then find out how much of the answer was memorised.
* docking — put the ligand in the pocket.